# DeepEval을 활용한 LLM 모델 평가


## 1. DeepEval 소개

DeepEval은 LLM(Large Language Model) 출력을 평가하기 위한 오픈소스 프레임워크입니다.


### 주요 특징

- **다양한 평가 메트릭**: Answer Relevancy, Faithfulness, Contextual Relevancy, G-Eval 등
- **실시간 평가**: LLM의 응답을 즉시 평가 가능
- **배치 평가**: 여러 테스트 케이스를 한 번에 평가
- **커스텀 메트릭**: 사용자 정의 평가 기준 생성 가능
- **다양한 LLM 지원**: OpenAI, HuggingFace, 로컬 모델 등


### 주의사항

1. **API 비용**: 
   - OpenAI API를 사용하므로 비용 발생
   - 작은 샘플로 먼저 테스트

2. **평가 모델 선택**:
   - GPT-4: 정확하지만 비용 높음
   - GPT-3.5-turbo: 빠르고 저렴하지만 정확도 낮음

3. **Threshold 설정**:
   - 도메인과 요구사항에 맞게 조정
   - 일반적으로 0.7~0.8 권장

4. **배치 크기**:
   - 너무 크면 시간과 비용 증가
   - 대표성 있는 샘플 선택


## 2. 환경 설정

- [Huggingface API Key](https://huggingface.co/settings/tokens)
- [OpenAI API Key](https://platform.openai.com/api-keys)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 3. 데이터 로딩

평가에 사용할 데이터셋을 로드합니다. CSV 파일에는 질문(`user_input`)과 정답(`reference`)이 포함되어 있습니다.


In [ ]:
import pandas as pd

# 데이터 로드
nutrition_df = pd.read_csv('data/nutrition.csv')
illnesses_df = pd.read_csv('data/illnesses.csv')

print(f"영양 데이터: {len(nutrition_df)}개")
print(f"질병 데이터: {len(illnesses_df)}개")

# 데이터 샘플 확인
print("\n=== 영양 데이터 샘플 ===")
print(nutrition_df.head(3))

print("\n=== 질병 데이터 샘플 ===")
print(illnesses_df.head(3))


## 4. DeepEval 주요 평가 메트릭

### Answer Relevancy (답변 관련성)
- **목적**: LLM의 답변이 질문과 얼마나 관련이 있는지 평가
- **점수 범위**: 0~1 (1에 가까울수록 좋음)
- **사용 사례**: 
    - 모든 Q&A 시스템에 필수
    - 모델이 질문에 정확히 답변했는지 확인


### Faithfulness (충실성)
- **목적**: 답변이 제공된 컨텍스트에 얼마나 충실한지 평가
- **점수 범위**: 0~1
- **사용 사례**: 
    - RAG 시스템에 필수
    - 모델이 환각(hallucination) 없이 사실에 기반한 답변을 하는지 확인


### Contextual Relevancy (컨텍스트 관련성)
- **목적**: 제공된 컨텍스트가 질문에 얼마나 관련이 있는지 평가
- **점수 범위**: 0~1
- **사용 사례**: 
    - RAG 시스템에서 검색된 문서가 적절한지 확인
    - 검색된 문서의 적절성 확인


### G-Eval (커스텀 평가)
- **목적**: 사용자 정의 기준으로 LLM 출력을 평가
- **점수 범위**: 사용자 정의
- **사용 사례**: 
    - 특정 도메인이나 요구사항에 맞는 평가 기준 적용
    - 의학, 법률 등 전문 분야

## 5. 실습: 단일 테스트 케이스 평가


### 테스트 케이스 생성

DeepEval에서는 `LLMTestCase`를 사용하여 평가할 데이터를 정의합니다.


In [ ]:
from deepeval.test_case import LLMTestCase

# 샘플 데이터 선택
sample = nutrition_df.iloc[0]

# 테스트 케이스 생성
test_case = LLMTestCase(
    input=sample['user_input'],
    expected_output=sample['reference'],
    # actual_output은 평가할 모델의 실제 출력을 넣어야 합니다
    actual_output="하루 1,000ml 이상 먹으면 안됩니다. 대부분의 아기들은 그보다 적게 먹는 것이 정상이며, 과도한 수유는 체중 증가나 이유식 문제를 일으킬 수 있습니다.",
    retrieval_context=[sample['reference']]  # RAG 시스템의 경우 검색된 컨텍스트
)

print("=== 테스트 케이스 ===")
print(f"질문: {test_case.input}")
print(f"\n기대 답변: {test_case.expected_output}")
print(f"\n실제 답변: {test_case.actual_output}")


### Answer Relevancy 평가

답변이 질문과 얼마나 관련이 있는지 평가합니다.


In [ ]:
from deepeval.metrics import AnswerRelevancyMetric

# Answer Relevancy 메트릭 생성
answer_relevancy = AnswerRelevancyMetric(
    threshold=0.7,  # 합격 기준 점수
    model="gpt-5o-nano",  # 평가에 사용할 LLM (gpt-3.5-turbo, gpt-4 등)
    include_reason=True  # 평가 이유 포함
)

# 평가 실행
answer_relevancy.measure(test_case)

print("=== Answer Relevancy 평가 결과 ===")
print(f"점수: {answer_relevancy.score:.3f}")
print(f"통과 여부: {'✓ 통과' if answer_relevancy.is_successful() else '✗ 실패'}")
print(f"\n평가 이유:\n{answer_relevancy.reason}")


### Faithfulness 평가

답변이 제공된 컨텍스트에 충실한지, 환각(hallucination)이 없는지 평가합니다.


In [ ]:
from deepeval.metrics import FaithfulnessMetric

# Faithfulness 메트릭 생성
faithfulness = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-5o-nano",
    include_reason=True
)

# 평가 실행
faithfulness.measure(test_case)

print("=== Faithfulness 평가 결과 ===")
print(f"점수: {faithfulness.score:.3f}")
print(f"통과 여부: {'✓ 통과' if faithfulness.is_successful() else '✗ 실패'}")
print(f"\n평가 이유:\n{faithfulness.reason}")


### G-Eval (커스텀 평가)

특정 도메인이나 요구사항에 맞는 커스텀 평가 기준을 정의할 수 있습니다.
소아과 영양 도메인에 특화된 평가 기준을 만들어보겠습니다.


In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

# 커스텀 평가 기준 정의
correctness_metric = GEval(
    name="Correctness",
    criteria="의학적 정확성 - 답변이 의학적으로 정확하고 안전한 정보를 제공하는지 평가",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    evaluation_steps=[
        "답변이 의학적으로 정확한 정보를 포함하는지 확인",
        "잘못된 의학 정보나 위험한 조언이 있는지 확인",
        "기대 답변과 비교하여 핵심 내용이 일치하는지 확인",
        "전문 용어를 올바르게 사용하는지 확인"
    ],
    threshold=0.7,
    model="gpt-5o-nano",
)

# 평가 실행
correctness_metric.measure(test_case)

print("=== Correctness (의학적 정확성) 평가 결과 ===")
print(f"점수: {correctness_metric.score:.3f}")
print(f"통과 여부: {'✓ 통과' if correctness_metric.is_successful() else '✗ 실패'}")
print(f"\n평가 이유:\n{correctness_metric.reason}")


## 6. 배치 평가 (Multiple Test Cases)

실제 모델 평가에서는 여러 테스트 케이스를 한 번에 평가해야 합니다.


### 모델 응답 시뮬레이션

실제로는 파인튜닝된 모델을 사용하지만, 여기서는 시뮬레이션 함수를 만들어보겠습니다.


In [ ]:
def get_model_response(question, reference):
    """
    실제 환경에서는 파인튜닝된 모델을 호출하는 함수입니다.
    여기서는 시뮬레이션을 위해 reference를 변형한 답변을 반환합니다.
    
    실제 사용 예시:
    # from transformers import pipeline
    # model = pipeline("text-generation", model="your-finetuned-model")
    # response = model(question)
    # return response[0]['generated_text']
    """
    # 시뮬레이션: reference의 일부를 사용
    # 실제로는 여기에 모델 호출 코드가 들어갑니다
    return reference[:100] + "..." if len(reference) > 100 else reference

# 테스트
sample_question = nutrition_df.iloc[0]['user_input']
sample_reference = nutrition_df.iloc[0]['reference']
response = get_model_response(sample_question, sample_reference)

print("질문:", sample_question)
print("\n모델 응답:", response)


### 여러 테스트 케이스 생성


In [ ]:
# 평가할 샘플 선택 (처음 10개)
test_samples = nutrition_df.head(10)

# 테스트 케이스 리스트 생성
test_cases = []

for idx, row in test_samples.iterrows():
    # 모델 응답 생성 (실제로는 파인튜닝된 모델 호출)
    model_output = get_model_response(row['user_input'], row['reference'])
    
    # 테스트 케이스 생성
    test_case = LLMTestCase(
        input=row['user_input'],
        actual_output=model_output,
        expected_output=row['reference'],
        retrieval_context=[row['reference']]
    )
    test_cases.append(test_case)

print(f"총 {len(test_cases)}개의 테스트 케이스 생성 완료!")


### 배치 평가 실행

`evaluate()` 함수를 사용하여 여러 테스트 케이스를 한 번에 평가할 수 있습니다.


In [ ]:
# 평가 메트릭 정의
metrics = [
    AnswerRelevancyMetric(threshold=0.7, model="gpt-5o-nano",),
    FaithfulnessMetric(threshold=0.7, model="gpt-5o-nano",),
    GEval(
        name="Correctness",
        criteria="의학적 정확성 평가",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
        threshold=0.7,
        model="gpt-5o-nano",
    )
]

# 배치 평가 실행
# 주의: 실제 실행 시 OpenAI API 키가 필요하고 비용이 발생합니다
# evaluate(test_cases=test_cases, metrics=metrics)

print("=== 배치 평가 설정 완료 ===")
print(f"테스트 케이스 수: {len(test_cases)}")
print(f"평가 메트릭 수: {len(metrics)}")
print("\n평가 메트릭:")
for i, metric in enumerate(metrics, 1):
    print(f"  {i}. {metric.__class__.__name__}")


## 7. HuggingFace 파인튜닝 모델과 통합


### 파이프라인 생성

In [ ]:
# HuggingFace 모델 로드 예시
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# 파인튜닝된 모델 로드
model_name = "your-finetuned-model-path"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 파이프라인 생성
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7
)


### 모델 응답 함수

In [ ]:
# 모델 응답 함수
def get_model_response(question):
    prompt = f"질문: {question}\n답변:"
    response = generator(prompt, max_new_tokens=256)[0]['generated_text']
    # 프롬프트 제거하고 답변만 추출
    answer = response.split("답변:")[-1].strip()
    return answer
    

### 테스트 케이스 생성 및 평가

In [ ]:
# 테스트 케이스 생성 및 평가
test_cases = []
for idx, row in test_samples.iterrows():
    model_output = get_model_response(row['user_input'])
    
    test_case = LLMTestCase(
        input=row['user_input'],
        actual_output=model_output,
        expected_output=row['reference']
    )
    test_cases.append(test_case)


### 평가 실행

In [ ]:
from deepeval import evaluate

# 평가 실행
evaluate(test_cases=test_cases, metrics=metrics)


## 8. 평가 결과 분석


### 결과 시각화


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 시뮬레이션된 평가 결과 (실제로는 evaluate() 함수의 결과를 사용)
evaluation_results = {
    'Answer Relevancy': [0.85, 0.78, 0.92, 0.88, 0.75, 0.90, 0.82, 0.87, 0.79, 0.91],
    'Faithfulness': [0.90, 0.85, 0.88, 0.92, 0.80, 0.87, 0.89, 0.91, 0.83, 0.86],
    'Correctness': [0.82, 0.79, 0.85, 0.88, 0.76, 0.84, 0.81, 0.86, 0.78, 0.87]
}

# 평균 점수 계산
avg_scores = {metric: np.mean(scores) for metric, scores in evaluation_results.items()}

# 막대 그래프
plt.figure(figsize=(10, 6))
metrics = list(avg_scores.keys())
scores = list(avg_scores.values())

plt.bar(metrics, scores, color=['#3498db', '#2ecc71', '#e74c3c'])
plt.axhline(y=0.7, color='r', linestyle='--', label='Threshold (0.7)')
plt.ylim(0, 1)
plt.ylabel('Average Score')
plt.title('모델 평가 메트릭별 평균 점수')
plt.legend()
plt.grid(axis='y', alpha=0.3)

for i, (metric, score) in enumerate(zip(metrics, scores)):
    plt.text(i, score + 0.02, f'{score:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\n=== 평가 메트릭별 평균 점수 ===")
for metric, score in avg_scores.items():
    status = "✓ 통과" if score >= 0.7 else "✗ 실패"
    print(f"{metric}: {score:.3f} {status}")


### 테스트 케이스별 상세 결과


In [ ]:
# 테스트 케이스별 결과 데이터프레임
results_df = pd.DataFrame({
    'Test Case': range(1, 11),
    'Answer Relevancy': evaluation_results['Answer Relevancy'],
    'Faithfulness': evaluation_results['Faithfulness'],
    'Correctness': evaluation_results['Correctness']
})

# 전체 점수 계산
results_df['Average'] = results_df[['Answer Relevancy', 'Faithfulness', 'Correctness']].mean(axis=1)
results_df['Pass'] = results_df['Average'] >= 0.7

print("=== 테스트 케이스별 평가 결과 ===\n")
print(results_df.to_string(index=False))

print(f"\n전체 통과율: {results_df['Pass'].sum()}/{len(results_df)} ({results_df['Pass'].sum()/len(results_df)*100:.1f}%)")


### 히트맵 시각화


In [ ]:
import seaborn as sns

# 히트맵 데이터 준비
heatmap_data = results_df[['Answer Relevancy', 'Faithfulness', 'Correctness']].T

# 히트맵 그리기
plt.figure(figsize=(12, 4))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn', 
            vmin=0, vmax=1, cbar_kws={'label': 'Score'},
            xticklabels=range(1, 11), 
            yticklabels=['Answer Relevancy', 'Faithfulness', 'Correctness'])
plt.xlabel('Test Case')
plt.ylabel('Metric')
plt.title('테스트 케이스별 평가 메트릭 히트맵')
plt.tight_layout()
plt.show()


### 과제 2: 커스텀 평가 메트릭 생성

질병 관련 응답의 안전성을 평가하는 커스텀 G-Eval 메트릭을 생성하세요.

**요구사항**:
1. 메트릭 이름: "Safety"
2. 평가 기준: 답변이 환자에게 안전한 정보를 제공하는지 평가
3. 평가 단계:
   - 위험한 자가 진단을 유도하지 않는지 확인
   - 의사 상담의 중요성을 강조하는지 확인
   - 정확한 의학 정보를 포함하는지 확인
4. Threshold: 0.8


In [ ]:
# 여기에 코드를 작성하세요

# Safety 메트릭 생성
safety_metric = GEval(
    name="Safety",
    criteria="",  # 여기에 평가 기준을 작성하세요
    evaluation_params=[
        # 필요한 파라미터를 추가하세요
    ],
    evaluation_steps=[
        # 평가 단계를 작성하세요
    ],
    threshold=0.8,
    model="gpt-4"
)

# 테스트 케이스 선택 및 평가


